# CatBoost Experiment — IEEE-CIS Fraud Detection

## 0. Setup & Imports

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'mlflow', 'dagshub', 'optuna', 'catboost', '--quiet'], capture_output=True)

import warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import mlflow, mlflow.sklearn
import dagshub, optuna
from catboost import CatBoostClassifier, Pool
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.base import BaseEstimator, TransformerMixin
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
print('CatBoost ready!')

In [ ]:
DAGSHUB_USERNAME = 'YOUR_DAGSHUB_USERNAME'
DAGSHUB_REPO     = 'YOUR_REPO_NAME'
dagshub.init(repo_owner=DAGSHUB_USERNAME, repo_name=DAGSHUB_REPO, mlflow=True)
mlflow.set_experiment('CatBoost_Training')

## 1. Data Loading

In [ ]:
BASE = '/kaggle/input/ieee-fraud-detection/'
train = pd.read_csv(BASE+'train_transaction.csv').merge(pd.read_csv(BASE+'train_identity.csv'), on='TransactionID', how='left')
test  = pd.read_csv(BASE+'test_transaction.csv').merge(pd.read_csv(BASE+'test_identity.csv'),  on='TransactionID', how='left')
print(train.shape, test.shape)

## 2. Cleaning

In [ ]:
with mlflow.start_run(run_name='CatBoost_Cleaning'):
    drop_missing = train.isnull().mean()[lambda x: x > 0.9].index.tolist()
    train.drop(columns=drop_missing, inplace=True)
    test.drop(columns=[c for c in drop_missing if c in test], inplace=True)

    # CatBoost handles missing values natively — we only drop extreme cases
    # Replace rare email domains
    for col in ['P_emaildomain', 'R_emaildomain']:
        if col in train:
            top = train[col].value_counts().nlargest(15).index
            train[col] = train[col].where(train[col].isin(top), 'other')
            test[col]  = test[col].where(test[col].isin(top), 'other')

    mlflow.log_param('dropped_high_missing', len(drop_missing))
    mlflow.log_metric('cols_remaining', train.shape[1])
    print(f'After cleaning: {train.shape}')

## 3. Feature Engineering

In [ ]:
with mlflow.start_run(run_name='CatBoost_Feature_Engineering'):

    def engineer(df):
        df = df.copy()
        df['hour']         = (df['TransactionDT'] / 3600) % 24
        df['day_of_week']  = (df['TransactionDT'] / (3600*24)) % 7
        df['is_night']     = ((df['hour'] >= 22) | (df['hour'] <= 6)).astype(int)
        df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
        df['TransactionAmt_log']    = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_cents']  = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
        df['TransactionAmt_isround']= (df['TransactionAmt_cents'] == 0).astype(int)
        if 'P_emaildomain' in df and 'R_emaildomain' in df:
            df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
        if 'card4' in df and 'card6' in df:
            df['card_combo'] = df['card4'].astype(str) + '_' + df['card6'].astype(str)
        for g in ['card1', 'card2']:
            if g in df:
                df[f'{g}_freq'] = df[g].map(df[g].value_counts())
        df['nan_count'] = df.isnull().sum(axis=1)
        return df

    train = engineer(train)
    test  = engineer(test)

    TARGET   = 'isFraud'
    DROP_COLS= ['TransactionID', 'TransactionDT', TARGET]

    # CatBoost handles categoricals directly — identify them
    cat_features = [c for c in train.select_dtypes(include='object').columns
                    if c not in DROP_COLS]

    # Fill NaN in cat cols with 'missing' string
    for col in cat_features:
        train[col] = train[col].fillna('missing').astype(str)
        test[col]  = test[col].fillna('missing').astype(str)

    mlflow.log_param('catboost_cat_features', len(cat_features))
    mlflow.log_metric('features_after_fe', train.shape[1])
    print(f'Shape after FE: {train.shape}')
    print(f'Categorical features: {len(cat_features)}')

## 4. Feature Selection

In [ ]:
with mlflow.start_run(run_name='CatBoost_Feature_Selection'):

    feature_cols = [c for c in train.columns if c not in DROP_COLS]
    X = train[feature_cols].copy()
    y = train[TARGET]

    # Fill numeric NaN for non-cat features
    num_cols = [c for c in feature_cols if c not in cat_features]
    X[num_cols] = X[num_cols].fillna(-999)

    cat_idx = [feature_cols.index(c) for c in cat_features if c in feature_cols]

    quick_cb = CatBoostClassifier(
        iterations=200, depth=6, learning_rate=0.1,
        eval_metric='AUC', task_type='GPU',
        random_seed=42, verbose=False
    )
    pool = Pool(X, y, cat_features=cat_idx)
    quick_cb.fit(pool)

    importances = pd.Series(quick_cb.get_feature_importance(), index=feature_cols)
    top_features = importances.nlargest(150).index.tolist()

    # Update cat_features to only include selected
    cat_features_sel = [c for c in cat_features if c in top_features]
    cat_idx_sel = [top_features.index(c) for c in cat_features_sel]

    fig, ax = plt.subplots(figsize=(10, 8))
    importances.nlargest(30).plot(kind='barh', ax=ax)
    ax.set_title('CatBoost Feature Importances (Top 30)')
    plt.tight_layout()
    plt.savefig('catboost_importance.png')
    mlflow.log_artifact('catboost_importance.png')

    mlflow.log_metric('features_selected', len(top_features))
    mlflow.log_metric('cat_features_selected', len(cat_features_sel))
    print(f'Selected {len(top_features)} features, {len(cat_features_sel)} categorical')

    X_sel      = train[top_features].copy()
    X_test_sel = test[top_features].copy()
    X_sel[num_cols]      = X_sel[[c for c in num_cols if c in top_features]].fillna(-999)
    X_test_sel[[c for c in num_cols if c in top_features]] = X_test_sel[[c for c in num_cols if c in top_features]].fillna(-999)

## 5. Training

### 5a. Underfitted — Very Shallow

In [ ]:
with mlflow.start_run(run_name='CatBoost_Underfitted'):
    params_u = dict(iterations=50, depth=2, learning_rate=0.5,
                    eval_metric='AUC', task_type='GPU', random_seed=42, verbose=False)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []
    for tr_i, val_i in cv.split(X_sel, y):
        m = CatBoostClassifier(**params_u)
        m.fit(Pool(X_sel.iloc[tr_i], y.iloc[tr_i], cat_features=cat_idx_sel))
        val_pred = m.predict_proba(Pool(X_sel.iloc[val_i], cat_features=cat_idx_sel))[:, 1]
        aucs.append(roc_auc_score(y.iloc[val_i], val_pred))
    mlflow.log_params(params_u)
    mlflow.log_metric('cv_auc_mean', np.mean(aucs))
    mlflow.log_param('note', 'intentionally_underfitted')
    print(f'[UNDERFITTED] CV AUC: {np.mean(aucs):.4f}')

### 5b. Overfitted — Deep, No Regularization

In [ ]:
with mlflow.start_run(run_name='CatBoost_Overfitted'):
    params_o = dict(iterations=3000, depth=10, learning_rate=0.3,
                    l2_leaf_reg=0.01, eval_metric='AUC',
                    task_type='GPU', random_seed=42, verbose=False)
    m_o = CatBoostClassifier(**params_o)
    m_o.fit(Pool(X_sel, y, cat_features=cat_idx_sel))
    train_auc = roc_auc_score(y, m_o.predict_proba(Pool(X_sel, cat_features=cat_idx_sel))[:, 1])

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    aucs = []
    for tr_i, val_i in cv.split(X_sel, y):
        m2 = CatBoostClassifier(**params_o)
        m2.fit(Pool(X_sel.iloc[tr_i], y.iloc[tr_i], cat_features=cat_idx_sel))
        val_pred = m2.predict_proba(Pool(X_sel.iloc[val_i], cat_features=cat_idx_sel))[:, 1]
        aucs.append(roc_auc_score(y.iloc[val_i], val_pred))

    mlflow.log_params(params_o)
    mlflow.log_metric('train_auc', train_auc)
    mlflow.log_metric('cv_auc', np.mean(aucs))
    mlflow.log_metric('overfit_gap', train_auc - np.mean(aucs))
    print(f'[OVERFITTED] Train: {train_auc:.4f} CV: {np.mean(aucs):.4f} Gap: {train_auc-np.mean(aucs):.4f}')

### 5c. Optuna Tuning

In [ ]:
def cb_objective(trial):
    params = {
        'iterations':    trial.suggest_int('iterations', 300, 2000),
        'depth':         trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'l2_leaf_reg':   trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'subsample':     trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),
        'min_data_in_leaf':  trial.suggest_int('min_data_in_leaf', 1, 50),
        'eval_metric': 'AUC', 'task_type': 'GPU',
        'random_seed': 42, 'verbose': False
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []
    for tr_i, val_i in cv.split(X_sel, y):
        m = CatBoostClassifier(**params)
        m.fit(Pool(X_sel.iloc[tr_i], y.iloc[tr_i], cat_features=cat_idx_sel))
        val_pred = m.predict_proba(Pool(X_sel.iloc[val_i], cat_features=cat_idx_sel))[:, 1]
        aucs.append(roc_auc_score(y.iloc[val_i], val_pred))
    return np.mean(aucs)

study = optuna.create_study(direction='maximize')
study.optimize(cb_objective, n_trials=20, show_progress_bar=True)
best_params = study.best_params
best_params.update({'eval_metric': 'AUC', 'task_type': 'GPU', 'random_seed': 42, 'verbose': False})
print(f'Best AUC: {study.best_value:.4f}')

### 5d. Final CV + Pipeline + Registry

In [ ]:
with mlflow.start_run(run_name='CatBoost_Final_CV'):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(y))
    test_preds = np.zeros(len(X_test_sel))
    fold_aucs = []

    for fold, (tr_i, val_i) in enumerate(cv.split(X_sel, y)):
        m = CatBoostClassifier(**best_params)
        m.fit(
            Pool(X_sel.iloc[tr_i], y.iloc[tr_i], cat_features=cat_idx_sel),
            eval_set=Pool(X_sel.iloc[val_i], y.iloc[val_i], cat_features=cat_idx_sel),
            early_stopping_rounds=50
        )
        val_pred = m.predict_proba(Pool(X_sel.iloc[val_i], cat_features=cat_idx_sel))[:, 1]
        oof[val_i] = val_pred
        test_preds += m.predict_proba(Pool(X_test_sel, cat_features=cat_idx_sel))[:, 1] / 5
        fa = roc_auc_score(y.iloc[val_i], val_pred)
        fold_aucs.append(fa)
        print(f'  Fold {fold+1}: {fa:.4f}')

    oof_auc = roc_auc_score(y, oof)
    mlflow.log_params(best_params)
    mlflow.log_metric('oof_auc', oof_auc)
    mlflow.log_metric('cv_auc_mean', np.mean(fold_aucs))
    print(f'OOF AUC: {oof_auc:.4f}')


class CatBoostPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, selected_features=None, cat_features=None):
        self.selected_features = selected_features
        self.cat_features_names = cat_features or []

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = self._engineer(X.copy())
        for c in self.cat_features_names:
            if c in df:
                df[c] = df[c].fillna('missing').astype(str)
        num_c = [c for c in df.columns if c not in self.cat_features_names]
        df[num_c] = df[num_c].fillna(-999)
        if self.selected_features:
            avail = [f for f in self.selected_features if f in df.columns]
            df = df[avail]
        return df

    def _engineer(self, df):
        df['hour']       = (df['TransactionDT'] / 3600) % 24
        df['day_of_week']= (df['TransactionDT'] / (3600*24)) % 7
        df['is_night']   = ((df['hour'] >= 22) | (df['hour'] <= 6)).astype(int)
        df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
        df['TransactionAmt_log']   = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_cents'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
        df['nan_count']  = df.isnull().sum(axis=1)
        if 'P_emaildomain' in df and 'R_emaildomain' in df:
            df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
        if 'card4' in df and 'card6' in df:
            df['card_combo'] = df['card4'].astype(str) + '_' + df['card6'].astype(str)
        return df


X_raw = train.drop(columns=['isFraud','TransactionID'], errors='ignore')
y_raw = train['isFraud']

cb_pipeline = Pipeline([
    ('preprocessor', CatBoostPreprocessor(selected_features=top_features, cat_features=cat_features_sel)),
    ('classifier',   CatBoostClassifier(**best_params))
])
cb_pipeline.fit(X_raw, y_raw)

with mlflow.start_run(run_name='CatBoost_Pipeline_Registry'):
    mlflow.log_metric('oof_auc', oof_auc)
    mlflow.sklearn.log_model(
        sk_model=cb_pipeline,
        artifact_path='catboost_fraud_pipeline',
        registered_model_name='CatBoost_Fraud_Pipeline'
    )
    print('CatBoost pipeline registered!')

np.save('catboost_test_preds.npy', test_preds)